## Create RDD （Resilient Distributed Dataset）

### 1. By using parallelize( ) function

In [11]:
# reference link: http://spark.apache.org/docs/latest/index.html

#set up SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder.master('local').appName('RDD_CreationApp').getOrCreate()
sc = spark.sparkContext

In [2]:
rdd = sc.parallelize([(1, 2, 3, 'a b c'),(4, 5, 6, 'd e f'),(7, 8, 9, 'g h i')])
print(type(rdd))
rdd.collect()

<class 'pyspark.rdd.RDD'>


[(1, 2, 3, 'a b c'), (4, 5, 6, 'd e f'), (7, 8, 9, 'g h i')]

In [3]:
#convert RDD to DataFrame
#df=rdd.toDF(['col1', 'col2', 'col3','col4'])
df=rdd.toDF()
df.show()
type(df)

+---+---+---+-----+
| _1| _2| _3|   _4|
+---+---+---+-----+
|  1|  2|  3|a b c|
|  4|  5|  6|d e f|
|  7|  8|  9|g h i|
+---+---+---+-----+



pyspark.sql.dataframe.DataFrame

In [4]:
myData = sc.parallelize([(1,2), (3,4), (5,6), (7,8), (9,10)])
print(type(myData))
myData.collect()

<class 'pyspark.rdd.RDD'>


[(1, 2), (3, 4), (5, 6), (7, 8), (9, 10)]

In [5]:
#convert RDD to DataFrame
mydf = myData.toDF()
print(type(mydf))
mydf.show()

<class 'pyspark.sql.dataframe.DataFrame'>
+---+---+
| _1| _2|
+---+---+
|  1|  2|
|  3|  4|
|  5|  6|
|  7|  8|
|  9| 10|
+---+---+



In [6]:
#convert DataFrame to RDD
rdd = mydf.rdd
print(type(rdd))
rdd.collect()

<class 'pyspark.rdd.RDD'>


[Row(_1=1, _2=2),
 Row(_1=3, _2=4),
 Row(_1=5, _2=6),
 Row(_1=7, _2=8),
 Row(_1=9, _2=10)]

### 2. By using createDataFrame( ) function

In [7]:
Employee = spark.createDataFrame([
    ('1', 'Joe', '70000', '1'),
    ('2', 'Henry', '80000', '2'),
    ('3', 'Sam', '60000', '2'),
    ('4', 'Max', '90000', '1')],
    ['Id', 'Name', 'Sallary','DepartmentId'])

Employee.show()


+---+-----+-------+------------+
| Id| Name|Sallary|DepartmentId|
+---+-----+-------+------------+
|  1|  Joe|  70000|           1|
|  2|Henry|  80000|           2|
|  3|  Sam|  60000|           2|
|  4|  Max|  90000|           1|
+---+-----+-------+------------+



### 3. By using read and load functions

#### a) Read dataset from .csv file

In [16]:
#df = spark.read.csv("california_cities.csv")
df = spark.read.csv("data/california_cities.csv", header=True)
df.show(5)
#df.printSchema()
rdd1 = df.rdd
list1 = rdd1.collect()
print(list1)


+-----------+-----------+------------+----------------+----------+---------+----------+------------------+
|       city|       latd|       longd|population_total|area_total|area_land|area_water|area_water_percent|
+-----------+-----------+------------+----------------+----------+---------+----------+------------------+
|   Adelanto|34.57611111|-117.4327778|           31765|   145.107|  145.062|     0.046|              0.03|
|AgouraHills|34.15333333|-118.7616667|           20330|     20.26|   20.184|     0.076|              0.37|
|    Alameda|37.75611111|-122.2744444|           75467|    59.465|   27.482|    31.983|             53.79|
|     Albany|37.88694444|-122.2977778|           18969|    14.155|    4.632|     9.524|             67.28|
|   Alhambra|34.08194444|    -118.135|           83089|    19.766|   19.763|     0.003|              0.01|
+-----------+-----------+------------+----------------+----------+---------+----------+------------------+
only showing top 5 rows

[Row(city='A

#### b) Read dataset from DataBase

In [12]:
## User information
user = 'changjiangzhang'
pw = '77981usA$2024'
## Database information
table_name = 'film'
url = 'jdbc:mysql://172.31.13.233:3306/sakila?user='+user+'&password='+pw
properties ={ 'password': pw,'user': user}
df = spark.read.jdbc(url=url, table=table_name,properties=properties)
df.show(10)
df.printSchema()

+-------+----------------+--------------------+------------+-----------+--------------------+---------------+-----------+------+----------------+------+--------------------+-------------------+
|film_id|           title|         description|release_year|language_id|original_language_id|rental_duration|rental_rate|length|replacement_cost|rating|    special_features|        last_update|
+-------+----------------+--------------------+------------+-----------+--------------------+---------------+-----------+------+----------------+------+--------------------+-------------------+
|      1|ACADEMY DINOSAUR|A Epic Drama of a...|  2006-01-01|          1|                NULL|              6|       0.99|    86|           20.99| PG   |Deleted Scenes,Be...|2006-02-14 21:03:42|
|      2|  ACE GOLDFINGER|A Astounding Epis...|  2006-01-01|          1|                NULL|              3|       4.99|    48|           12.99| G    |Trailers,Deleted ...|2006-02-14 21:03:42|
|      3|ADAPTATION HOLES|A As

In [10]:
from pyspark import SparkContext
from pyspark.sql import SQLContext

#you should copy mysql-connector-java-8.0.27.jar to spark-3.1.2-bin-hadoop3.2/jars
#otherwise, you will got an error: no suitable driver

df = spark.read.format("jdbc").options(url\
        ="jdbc:mysql://127.0.0.1:3306/uic2?user=cnuser&password=taffydog",\
        dbtable="student").load()
df.show()


+-----+--------+----------+--------+
|   ID|    name| dept_name|tot_cred|
+-----+--------+----------+--------+
|11111|    test|     Music|      10|
|12345| Shankar|Comp. Sci.|      32|
|12888|    Jack| Data Sci.|     110|
|19991|  Brandt|   History|      80|
|23121|  Chavez|   Finance|     110|
|33333|     Tom| Data Sci.|     120|
|44553| Peltier|   Physics|      56|
|45678|    Levy|   Physics|      46|
|54321|Williams|Comp. Sci.|      54|
|55739| Sanchez|     Music|     150|
|70557|    Snow|   Physics|       0|
|76543|   Brown|Comp. Sci.|      58|
|76653|     Aoi|Elec. Eng.|      60|
|98765|Bourikas|Elec. Eng.|      98|
|98988|  Tanaka|   Biology|     128|
+-----+--------+----------+--------+



#### c) Read dataset from HDFS

In [ ]:
# you can do this lab exercise in your local Windows, Mac or Linux system, 
# or Cloud systems
# in order to load data in hdfs format, you need to start hadoop first
# i.e. start-dfs.cmd and start-yarn.cmd in Windows, or start-dfs.sh and start-yarn.sh in Linux system
# make sure hadoop system start correctly
# create a directory in hdfs:  hdfs dfs -mkdir /hdfs
# upload california_cities.csv to hdfs: 
#     hdfs dfs -put california_cities.csv  /hdfs
#  or for Windows 10, suppose your file is in the folder E:\UIC2022_Spring\DPWII\spark_lab\
#  you can type th ecommand:
#     hdfs dfs -put E:\UIC2022_Spring\DPWII\spark_lab\california_cities.csv /hdfs/
# check the file is uploaded: hdfs dfs -ls /hdfs

rdd1 = sc.textFile("hdfs://localhost:9000/hdfs/california_cities.csv")
rdd1 = rdd1.map(lambda x: x.split(','))
list1 = rdd1.collect()
print(list1)


In [ ]:
# to Pandas DataFrame
header =  ['city','latd','longd','population_total','area_total','area_land','area_water','area_water_percent']
import pandas as pd
df = pd.DataFrame(list1, columns = header)
print(df)

In [ ]:
# use default column name
type(rdd1)
df1 = rdd1.toDF()
#df1.show()

In [ ]:
# to Spark DataFrame using toDF(), we provide the header to this DataFrame
# for this case, rdd1 has a column name in it, 
# as an exercise, can you create a dataframe by using the first row of rdd1 as the header ?
df1 = rdd1.toDF(header)
df1.show()

In [ ]:
# to Spark DataFrame using createDataFrame 
df1 = spark.createDataFrame(rdd1, header) 
df1.show()

In [ ]:
sc.stop()
spark.stop()